# J3 PM | Traitement de données audio. À bon entendeur...

**Objectifs d'apprentissage :**
- Transcrire des fichiers audio directement depuis le web sans les télécharger sur le disque.
- Utiliser un LLM pour nettoyer et formater un entretien qualitatif brut.
- Construire une boucle d'automatisation pour traiter un corpus de plusieurs audios (Topic Modeling).


In [ ]:
from openai import OpenAI

# 1. Coller votre clé API d'OPEN AI
api_key = "sk-votre-cle-secrete-ici"

try:
    client = OpenAI(api_key=api_key)
    print("Client OpenAI initialisé avec succès !")
except Exception as e:
    print("Erreur d'initialisation :", e)


## 1. Retranscription : Un cas en français

Analysons un discours de campagne en français. Pour cet exercice, nous allons retranscrire un extrait d'un meeting de campagne présidentielle de Jean-Luc Mélenchon en utilisant la même méthode.
*(Source : [Chaîne YouTube officielle, Meeting de campagne présidentielle 2022](https://www.youtube.com/watch?v=x77tDojxgFU))*


In [ ]:
import requests
from io import BytesIO


In [ ]:
# Extrait campagne de Jean-Luc Mélenchon en français
melenchon_file = "melenchon_campagne.mp3"
fr_audio_url = f"https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/resources/j3/2_aprem/audio/{melenchon_file}"

# Téléchargement du file_item audio depuis GitHub
response_fr = requests.get(fr_audio_url)

# BytesIO simule un file_item en mémoire vive, évitant de devoir l'enregistrer sur le disque dur
# Certaines API s'attendent à lire un file_item avec un nom explicite (pour déduire le format audio)
audio_fr = BytesIO(response_fr.content)
audio_fr.name = melenchon_file

In [ ]:
# Demande de retranscription à l'IA
transcript_fr = client.audio.transcriptions.create(
    model="whisper-1",
    file=audio_fr
)

print("--- Retranscription (Français) ---")
print(transcript_fr.text)


## 2. Retranscription : Une autre langue...

Nous allons charger un court extrait d'un State of the Union (SOTU) de Donald Trump directement en mémoire, puis utiliser Whisper pour la retranscription en anglais.
*(Source : [Archive publique YouTube, State of the Union Address 2018](https://www.youtube.com/watch?v=jt1gBx4Fu5U))*


In [ ]:
# Extrait d'un discours de Donald Trump en anglais (State of the Union court)
trump_file = "trump_sotu.mp3"
trump_audio_url = f"https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/resources/j3/2_aprem/audio/{trump_file}"

# Téléchargement du file_item depuis notre dépôt GitHub
response = requests.get(trump_audio_url)

# Chargement direct en mémoire pour une exécution plus rapide
audio = BytesIO(response.content)
audio.name = trump_file


In [ ]:
# Appel à l'API Whisper d'OpenAI pour retranscrire l'audio en texte
transcript_trump = client.audio.transcriptions.create(
    model="whisper-1",
    file=audio,
)

print("--- Retranscription (Anglais) ---")
print(transcript_trump.text)


## 3. Retranscription : Analyse d'un pseudo entretien...

L'analyse d'entretiens semi-directifs est au cœur de nombreuses recherches qualitatives. Pour illustrer notre méthode, nous utiliserons un court extrait d'une **interview publique du sociologue Pierre Bourdieu** simulant un enregistrement de terrain.
*(Source : [Extrait d'interview télévisée, archives YouTube](https://www.youtube.com/watch?v=um7yAToVwcY))*


In [ ]:
# On télécharge ...
interview_file = "interview.mp3"
interview_audio_url = f"https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/resources/j3/2_aprem/audio/{interview_file}"

response_interview = requests.get(interview_audio_url)

audio_interview = BytesIO(response_interview.content)
audio_interview.name = interview_file


In [ ]:
# Retranscription automatique de l'entretien 
transcript_interview = client.audio.transcriptions.create(
    model="whisper-1",
    file=audio_interview,
)

print("--- Retranscription brute (Interview) ---")
print(transcript_interview.text)


### 3.1 Restructuration par un LLM (~Prompt Engineering)

Une transcription Whisper brute comporte souvent de nombreuses hésitations orales ("euh", bégaiements). Nous allons utiliser le LLM pour transformer cet output en un format exploitable (Enquêteur / Enquêté) et corriger les scories de l'oralité.


In [ ]:
system_prompt = '''
Tu es sociologue.
Voici une transcription brute d'un entretien qualitatif.
Ton rôle est de :
1. Nettoyer les hésitations (les "euh", les mots répétés) pour rendre la lecture fluide.
2. Ajouter des sauts de ligne entre chaque prise de parole.
'''


In [ ]:
# Envoi de la consigne et du texte brut au modèle GPT-4o-mini pour nettoyage
clean_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Voici la retranscription brute :\n\n{transcript_interview.text}"},
    ],
    temperature=0.2
)

clean_interview = clean_response.choices[0].message.content
print("--- Entretien nettoyé et structuré ---")
print(clean_interview)


### 3.2 Exporter vos retranscriptions
Une fois le nettoyage terminé, vous souhaiterez sûrement conserver ce texte pour vos archives ou pour l'importer dans un logiciel d'analyse textuelle (NVivo, IRAMUTEQ...). Voici comment le sauvegarder en un clin d'œil dans un simple fichier `.txt` :


In [ ]:
# Sauvegarde de notre entretien nettoyé
export_filename = "entretien_bourdieu_clean.txt"

with open(export_filename, "w", encoding="utf-8") as file_item:
    file_item.write(clean_interview)

print(f"Le texte a été sauvegardé avec succès dans le file_item : {export_filename}")

## 4. Automatisation ? DRY!

En recherche, nous avons souvent des dizaines d'audios à traiter. Plutôt que de répéter le code manuellement, l'idéal est de créer une **fonction réutilisable** qui effectue tout le travail (téléchargement, retranscription et extraction des thèmes). Nous pourrons ensuite appeler cette fonction sur tous les fichiers de notre corpus grâce à une simple boucle !


In [ ]:
def analyze_audio_themes(audio_url):
    """
    Cette fonction prend en entrée l'URL d'un file_item audio en ligne, 
    le télécharge en mémoire, le retranscrit avec Whisper, 
    et extrait ses grands thèmes avec GPT-4o-mini.
    """

    # 0. Un peu de feedback.
    filename = audio_url.split("/")[-1]
    print(f"\n========== TRAITEMENT DE : {filename} ==========")

    # 1. Téléchargement direct en mémoire depuis n'importe quelle URL
    response = requests.get(audio_url)
    
    # On utilise BytesIO pour ne pas encombrer le disque dur de la machine virtuelle
    audio_buffer = BytesIO(response.content)
    # Astuce : On récupère le nom du file_item à la fin de l'URL pour que Whisper détecte le format (.mp3)
    audio_buffer.name = audio_url.split("/")[-1]
    
    # 2. Retranscription avec Whisper
    # 1er appel API : Retranscription du contenu audio
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_buffer,
    )
    raw_text = transcript.text
    print(f"[+] Retranscription de {audio_buffer.name} terminée ({len(raw_text)} caractères).")
    
    # 3. Extraction thématique avec le LLM
    system_prompt_themes = '''
    Tu es un chercheur en sciences sociales.
    Identifie les 2 thèmes principaux abordés dans ce court extrait.
    Pour chaque thème, donne simplement un titre court et une phrase d'explication.
    '''
    
    # 4. Nouvel appel API : Extraction du sens à partir du texte généré précédemment
    response_themes = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt_themes},
            {"role": "user", "content": raw_text},
        ],
        temperature=0.3
    )
    
    return response_themes.choices[0].message.content

Maintenant que notre fonction est prête, il suffit d'itérer sur notre liste de fichiers pour tout automatiser.


In [ ]:
# Liste des URLs de nos fichiers audio (notre "corpus")
audio_urls = [
    "https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/resources/j3/2_aprem/audio/trump_sotu.mp3", 
    "https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/resources/j3/2_aprem/audio/melenchon_campagne.mp3", 
    "https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/resources/j3/2_aprem/audio/interview.mp3",
]

for url in audio_urls:
    # On passe directement l'URL à notre fonction !
    themes = analyze_audio_themes(url)
    print("[+] Thèmes identifiés :")
    print(themes)
    print("====================================================")

## Hack-time 
Comment pouvons nous ajouter un paramètre pour modifier le prompt system?


## 5. Amélioration des retranscription (Post-Processing LLM)

Bien que Whisper soit performant, la retranscription brute manque souvent de ponctuation expressive, contient des bégaiements, et peut faire des erreurs phonétiques (notamment sur les noms propres ou les acronymes spécifiques à la politique).

Nous allons construire un **Agent Transcripteur Expert** avec un "System Prompt" très strict pour nettoyer et améliorer le discours de Jean-Luc Mélenchon (ou tout autre discours brut), tout en conservant scrupuleusement le style et le sens d'origine.


In [ ]:
expert_transcriber_prompt = '''
Tu es un transcripteur professionnel spécialisé en sciences sociales et en discours politiques.
Ta tâche est de corriger une transcription audio brute générée par une IA.

Instructions strictes :
1. Corrige les fautes d'orthographe, de grammaire et de ponctuation induites par l'IA (homophones, erreurs de découpage de phrases).
2. Supprime les bégaiements excessifs et les tics de langage inutiles ("euh", "ben", "alors voilà"), SAUF si cela modifie le sens ou l'intention de l'interlocuteur.
3. Rétablis les noms propres, acronymes ou termes techniques qui auraient pu être mal compris phonétiquement.
4. Ajoute une ponctuation expressive qui reflète le rythme et le ton (points d'exclamation, interrogations, pauses avec des virgules).
5. Ne modifie JAMAIS le vocabulaire choisi, garde le style et l'authenticité exacts de l'orateur.
'''


In [ ]:
# On utilise la retranscription brute de la partie 1
raw_text_to_fix = transcript_fr.text 

# On demande à GPT-4o-mini de corriger le texte
correction_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": expert_transcriber_prompt},
        {"role": "user", "content": f"Voici la transcription brute à corriger :\n\n{raw_text_to_fix}"},
    ],
    temperature=0.2 # Température basse car on veut de la rigueur, pas de la créativité
)

fixed_text = correction_response.choices[0].message.content


Pour bien comprendre l'apport du LLM, affichons les deux versions côte à côte (ou l'une en dessous de l'autre) :


In [ ]:
print("==============================================================================")
print("❌ AVANT : Retranscription Whisper Brute (Sans ponctuation, erreurs possibles)")
print("==============================================================================")
print(raw_text_to_fix)
print("\n\n")
print("==============================================================================")
print("✅ APRÈS : Correction par le LLM (Ponctuation rétablie, sens clarifié)")
print("==============================================================================")
print(fixed_text)

## Conclusion 

Nous avons vu comment automatiser le traitement d'enregistrements audio. 

La retranscription automatique (avec Whisper) permet un gain de temps considérable par rapport au fastidieux travail de retranscription manuelle. L'utilisation d'un LLM dans un second temps pour nettoyer le texte ou extraire des thèmes (Topic Modeling) est facultative mais peut être intéressante.

**Points clés à retenir pour vos recherches :**
1. **Rapport taille/performance** : Notez que nous utilisons ici de tout petits modèles (comme GPT-4o-mini ou le modèle de base de Whisper). Ils offrent déjà d'excellentes performances pour ces tâches sans nécessiter d'infrastructures lourdes.
2. **Reproductibilité** : Le fait de scripter ces étapes apporte de la transparence sur les processus.
3. **Perte d'authenticité** : Le nettoyage par un LLM (et même la retranscription brute) implique toujours un risque de modification du sens, de lissage excessif du langage ou d'erreurs phonétiques. L'IA ne remplace pas votre écoute : le chercheur doit confronter le texte à l'audio d'origine.